## Data Loading

In [1]:
import pandas as pd
df = pd.read_csv('data/results.csv')

In [2]:
# See the first 5 rows
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
# How many rows and columns
df.shape

(49287, 9)

In [4]:
# Column names and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49287 entries, 0 to 49286
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49287 non-null  str    
 1   home_team   49287 non-null  str    
 2   away_team   49287 non-null  str    
 3   home_score  49215 non-null  float64
 4   away_score  49215 non-null  float64
 5   tournament  49287 non-null  str    
 6   city        49287 non-null  str    
 7   country     49287 non-null  str    
 8   neutral     49287 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [5]:
# Basic statistics
df.describe()

,home_score,away_score
count,49215.000000,49215.000000
mean,1.756091,1.182404
std,1.770617,1.401770
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


In [6]:
# Check for missing values
df.isnull().sum()

date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

## Data Cleaning

In [7]:
# remove rows that contains cells with empty values
df.dropna(inplace=True)
df.shape

(49215, 9)

In [8]:
# convert date format from strings to date_time
df['date'] = pd.to_datetime(df['date'])

In [9]:
# filter rows by date (after 2010)
filtered_df = df[df['date'] > '2010-01-01']
filtered_df.shape

(15632, 9)

In [10]:
# filter rows by tournament type
tournament = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'African Cup of Nations',
    'African Cup of Nations qualification',
    'AFC Asian Cup',
    'AFC Asian Cup qualification',
    'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
    'Copa América',
    'Copa América qualification',
    'UEFA Nations League',
    'AFF Championship'
]

filtered_df = filtered_df[filtered_df['tournament'].isin(tournament)]
filtered_df.shape

(8202, 9)

In [11]:
# fix team name mismatches in filtered_df to match world cup teams
filtered_df['home_team'] = filtered_df['home_team'].replace({
    'Czech Republic': 'Czechia',
    'United States': 'USA'
})

filtered_df['away_team'] = filtered_df['away_team'].replace({
    'Czech Republic': 'Czechia',
    'United States': 'USA'
})

In [12]:
# add a new column called results
import numpy as np

conditions = [
    (filtered_df['home_score'] > filtered_df['away_score']),
    (filtered_df['home_score'] == filtered_df['away_score']),
    (filtered_df['home_score'] < filtered_df['away_score']),
]

results = ['home_win', 'draw', 'away_win']

filtered_df['results'] = np.select(conditions, results, default='unknown')

## Feature Engineering

In [13]:
# convert boolean (True/False) to 1/0
#filtered_df['neutral'] = filtered_df['neutral'].replace({True: 1, False: 0})
filtered_df['neutral'] = filtered_df['neutral'].astype(int)

In [14]:
# tournament encoder

filtered_df['tournament'] = filtered_df['tournament'].astype('category')
filtered_df['tournament_encode'] = filtered_df['tournament'].cat.codes

In [15]:
# calculate head-to-head record between home and away team

def get_h2h(row, filtered_df):
    home_team = row['home_team']
    away_team = row['away_team']

    previous_matches = filtered_df[
        (
            ((filtered_df['home_team'] == home_team) & (filtered_df['away_team'] == away_team)) |
            ((filtered_df['home_team'] == away_team) & (filtered_df['away_team'] == home_team))
        ) &
        (filtered_df['date'] < row['date'])
    ]

    h2h_home_win = len(previous_matches[
        ((previous_matches['results'] == 'home_win') & (previous_matches['home_team'] == home_team)) |
        ((previous_matches['results'] == 'away_win') & (previous_matches['away_team'] == home_team))
    ])

    h2h_away_win = len(previous_matches[
        ((previous_matches['results'] == 'home_win') & (previous_matches['home_team'] == away_team)) |
        ((previous_matches['results'] == 'away_win') & (previous_matches['away_team'] == away_team))
    ])

    h2h_draw = len(previous_matches[(previous_matches['results'] == 'draw')])

    return h2h_home_win, h2h_away_win, h2h_draw


filtered_df[['h2h_home_win', 'h2h_away_win', 'h2h_draw']] = filtered_df.apply(get_h2h, axis=1, args=(filtered_df,), result_type='expand')

In [16]:
# calculate recent form (win rate from last 5 matches) for home and away team

def recent_form(row, filtered_df):
    home_team = row['home_team']
    away_team = row['away_team']

    home_previous_matches = filtered_df[
        (
            ((filtered_df['home_team'] == home_team) | (filtered_df['away_team'] == home_team))
        ) &
        (filtered_df['date'] < row['date'])
    ].sort_values(by='date').tail(5)

    away_previous_matches = filtered_df[
        (
            ((filtered_df['home_team'] == away_team) | (filtered_df['away_team'] == away_team))
        ) &
        (filtered_df['date'] < row['date'])
    ].sort_values(by='date').tail(5)

    home_team_wins = len(home_previous_matches[
        ((home_previous_matches['results'] == 'home_win') & (home_previous_matches['home_team'] == home_team)) |
        ((home_previous_matches['results'] == 'away_win') & (home_previous_matches['away_team'] == home_team))
    ])

    away_team_wins = len(away_previous_matches[
        ((away_previous_matches['results'] == 'home_win') & (away_previous_matches['home_team'] == away_team)) |
        ((away_previous_matches['results'] == 'away_win') & (away_previous_matches['away_team'] == away_team))
    ])

    home_form = home_team_wins / len(home_previous_matches) if len(home_previous_matches) > 0 else 0
    away_form = away_team_wins / len(away_previous_matches) if len(away_previous_matches) > 0 else 0

    return home_form, away_form

filtered_df[['home_form', 'away_form']] = filtered_df.apply(recent_form, axis=1, args=(filtered_df,), result_type='expand')

In [17]:
filtered_df[['home_team', 'away_team', 'home_form', 'away_form']].tail(20)

,home_team,away_team,home_form,away_form
49062,Senegal,Egypt,0.8,0.8
49063,Morocco,Nigeria,0.8,1.0
49064,Egypt,Nigeria,0.6,0.8
49065,Morocco,Senegal,0.6,0.8
49084,New Caledonia,Jamaica,0.6,0.4
49085,Bolivia,Suriname,0.4,0.4
49086,Italy,Northern Ireland,0.8,0.4
49087,Wales,Bosnia and Herzegovina,0.6,0.4
49088,Poland,Albania,0.6,0.6
49089,Ukraine,Sweden,0.6,0.0


In [18]:
# fifa ranking data loading
fifa_rankings = pd.read_csv('data/fifa_rankings.csv')

In [19]:
fifa_rankings.tail(10)

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
67251,145.0,Ethiopia,ETH,1068.48,1068.79,0,CAF,2024-04-04
67252,144.0,Suriname,SUR,1071.85,1071.85,0,CONCACAF,2024-04-04
67253,143.0,Turkmenistan,TKM,1072.52,1078.25,2,AFC,2024-04-04
67254,142.0,Antigua and Barbuda,ATG,1072.66,1072.66,0,CONCACAF,2024-04-04
67255,141.0,Philippines,PHI,1075.14,1086.17,2,AFC,2024-04-04
67256,140.0,Burundi,BDI,1081.63,1085.83,0,CAF,2024-04-04
67257,139.0,Kuwait,KUW,1085.46,1094.05,2,AFC,2024-04-04
67258,138.0,Malaysia,MAS,1094.54,1110.17,6,AFC,2024-04-04
67259,162.0,Tahiti,TAH,999.48,999.48,-1,OFC,2024-04-04
67260,5.0,Brazil,BRA,1788.65,1784.09,0,CONMEBOL,2024-04-04


In [20]:
# convert rank_date to datetime
fifa_rankings['rank_date'] = pd.to_datetime(fifa_rankings['rank_date'])

In [21]:
# fix team name mismatches between fifa_rankings and filtered_df
fifa_rankings['country_full'] = fifa_rankings['country_full'].replace({
    'Congo DR': 'DR Congo',
    'IR Iran': 'Iran',
    'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',
    "Côte d'Ivoire": 'Ivory Coast',
    'Curacao': 'Curaçao',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Sao Tome and Principe': 'São Tomé and Príncipe',
    'Brunei Darussalam': 'Brunei',
    'The Gambia': 'Gambia',
    'St Kitts and Nevis': 'Saint Kitts and Nevis',
    'St Lucia': 'Saint Lucia',
    'St Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Cabo Verde': 'Cape Verde'
})

In [22]:
# sort both dataframes by date (required for merge_asof)
filtered_df = filtered_df.sort_values(by='date', ascending=True)
fifa_rankings = fifa_rankings.sort_values(by='rank_date', ascending=True)

In [23]:
# merge_asof for home team
filtered_df = pd.merge_asof(left=filtered_df, right=fifa_rankings, left_on='date', right_on='rank_date', left_by='home_team', right_by='country_full', direction='backward')

In [24]:
# rename column rank to home_rank
filtered_df = filtered_df.rename(columns={"rank": "home_rank"})

In [25]:
# drop unnecessary columns (for merge 1 - home)
filtered_df = filtered_df.drop(columns=['country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date'])

In [26]:
# check current filtered_df
filtered_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'results',
       'tournament_encode', 'h2h_home_win', 'h2h_away_win', 'h2h_draw',
       'home_form', 'away_form', 'home_rank'],
      dtype='str')

In [27]:
# merge_asof for away team
filtered_df = pd.merge_asof(left=filtered_df, right=fifa_rankings, left_on='date', right_on='rank_date', left_by='away_team', right_by='country_full', direction='backward')

In [28]:
# rename column rank to away_rank
filtered_df = filtered_df.rename(columns={"rank": "away_rank"})

In [29]:
# drop unnecessary columns (for merge 2 - away)
filtered_df = filtered_df.drop(columns=['country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date'])

In [30]:
# check current filtered_df
filtered_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'results',
       'tournament_encode', 'h2h_home_win', 'h2h_away_win', 'h2h_draw',
       'home_form', 'away_form', 'home_rank', 'away_rank'],
      dtype='str')

In [31]:
# find rank difference (home - away)
filtered_df['rank_difference'] = filtered_df['home_rank'] - filtered_df['away_rank']

In [32]:
# verify rank difference results
filtered_df[['home_team', 'away_team', 'home_rank', 'away_rank', 'rank_difference']].head(10)

,home_team,away_team,home_rank,away_rank,rank_difference
0,Bahrain,Hong Kong,60.0,143.0,-83.0
1,United Arab Emirates,Malaysia,112.0,160.0,-48.0
2,Thailand,Jordan,105.0,111.0,-6.0
3,Singapore,Iran,110.0,64.0,46.0
4,Yemen,Japan,130.0,43.0,87.0
5,Kuwait,Australia,104.0,21.0,83.0
6,Indonesia,Oman,120.0,79.0,41.0
7,China PR,Syria,93.0,91.0,2.0
8,Lebanon,Vietnam,148.0,123.0,25.0
9,Angola,Mali,95.0,47.0,48.0


In [33]:
# check for missing rank difference values
filtered_df['rank_difference'].isnull().sum()

np.int64(220)

In [34]:
filtered_df = filtered_df.dropna(subset=['rank_difference'])
filtered_df.shape

(7982, 19)

In [35]:
# check final columns after all feature engineering
filtered_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'results',
       'tournament_encode', 'h2h_home_win', 'h2h_away_win', 'h2h_draw',
       'home_form', 'away_form', 'home_rank', 'away_rank', 'rank_difference'],
      dtype='str')

In [36]:
# check final shape after all feature engineering
filtered_df.shape

(7982, 19)

In [37]:
filtered_df.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,results,tournament_encode,h2h_home_win,h2h_away_win,h2h_draw,home_form,away_form,home_rank,away_rank,rank_difference
8192,2026-03-26,Slovakia,Kosovo,3.0,4.0,FIFA World Cup qualification,Bratislava,Slovakia,0,away_win,10,0,0,0,0.6,0.6,48.0,102.0,-54.0
8193,2026-03-26,Turkey,Romania,1.0,0.0,FIFA World Cup qualification,Istanbul,Turkey,0,home_win,10,1,1,0,0.6,0.6,40.0,46.0,-6.0
8194,2026-03-26,Czechia,Republic of Ireland,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,0,draw,10,0,0,0,0.4,0.6,36.0,60.0,-24.0
8195,2026-03-26,Denmark,North Macedonia,4.0,0.0,FIFA World Cup qualification,Copenhagen,Denmark,0,home_win,10,0,0,0,0.6,0.4,21.0,69.0,-48.0
8196,2026-03-31,Sweden,Poland,3.0,2.0,FIFA World Cup qualification,Solna,Sweden,0,home_win,10,1,1,0,0.2,0.8,27.0,28.0,-1.0
8197,2026-03-31,Bosnia and Herzegovina,Italy,1.0,1.0,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,0,draw,10,0,3,1,0.2,0.8,74.0,9.0,65.0
8198,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,Pristina,Kosovo,0,away_win,10,0,2,0,0.6,0.8,102.0,40.0,62.0
8199,2026-03-31,DR Congo,Jamaica,1.0,0.0,FIFA World Cup qualification,Zapopan,Mexico,1,home_win,10,0,0,0,0.4,0.4,63.0,55.0,8.0
8200,2026-03-31,Iraq,Bolivia,2.0,1.0,FIFA World Cup qualification,Guadalupe,Mexico,1,home_win,10,0,0,0,0.6,0.6,58.0,85.0,-27.0
8201,2026-03-31,Czechia,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,0,draw,10,0,2,1,0.4,0.6,36.0,21.0,15.0


## Model Training

In [38]:
# split data (train-test = 80/20)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

x = filtered_df[['neutral', 'tournament_encode', 'h2h_home_win', 'h2h_away_win', 'h2h_draw', 'home_form', 'away_form', 'home_rank', 'away_rank', 'rank_difference']]
y = filtered_df['results']

le = LabelEncoder()
y = le.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

x_train shape: (6385, 10)
x_test shape: (1597, 10)
y_train shape: (6385,)
y_test shape: (1597,)


In [39]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(x_train, y_train)

predictions_lr = model_lr.predict(x_test)
accuracy_lr = model_lr.score(x_test, y_test)

print(f"Logistic Regression accuracy: {accuracy_lr:.2%}")

Logistic Regression accuracy: 58.86%


c:\Users\admir\UK\Personal Projects\World Cup 2026 AI Prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [40]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(x_train, y_train)

predictions_rf = model_rf.predict(x_test)
accuracy_rf = model_rf.score(x_test, y_test)

print(f"Random Forest accuracy: {accuracy_rf:.2%}")

Random Forest accuracy: 56.42%


In [41]:
# XGBoost
import xgboost as xgb
from xgboost import XGBClassifier

model_xgb = xgb.XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=3)
model_xgb.fit(x_train, y_train)

predictions_xgb = model_xgb.predict(x_test)
accuracy_xgb = model_xgb.score(x_test, y_test)

print(f"XGBoost accuracy: {accuracy_xgb:.2%}")

XGBoost accuracy: 58.74%


In [42]:
# compare all three models
print(f"Logistic Regression accuracy: {accuracy_lr:.2%}")
print(f"Random Forest accuracy:       {accuracy_rf:.2%}")
print(f"XGBoost accuracy:             {accuracy_xgb:.2%}")
print(f"\nBest model: XGBoost with {accuracy_xgb:.2%} accuracy")

Logistic Regression accuracy: 58.86%
Random Forest accuracy:       56.42%
XGBoost accuracy:             58.74%

Best model: XGBoost with 58.74% accuracy


In [43]:
# save model
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model_xgb, 'models/xgb_model.joblib')

['models/xgb_model.joblib']

## Tournament Simulator

In [44]:
# precompute recent form
from world_cup_teams import rankings
from world_cup_teams import groups

all_teams = [team for group in groups.values() for team in group]
team_form = {}

for team in all_teams:
    team_matches = filtered_df[
        (filtered_df['home_team'] == team) | (filtered_df['away_team'] == team)
    ].sort_values(by='date').tail(5)
    wins = len(team_matches[
        ((team_matches['results'] == 'home_win') & (team_matches['home_team'] == team)) |
        ((team_matches['results'] == 'away_win') & (team_matches['away_team'] == team))
    ])
    team_form[team] = wins / len(team_matches) if len(team_matches) > 0 else 0

In [45]:
# precompute h2h
from itertools import combinations

h2h_dict = {}
for home_team, away_team in combinations(all_teams, 2):
    previous_matches = filtered_df[
        ((filtered_df['home_team'] == home_team) & (filtered_df['away_team'] == away_team)) |
        ((filtered_df['home_team'] == away_team) & (filtered_df['away_team'] == home_team))
    ]
    home_wins = len(previous_matches[
        ((previous_matches['results'] == 'home_win') & (previous_matches['home_team'] == home_team)) |
        ((previous_matches['results'] == 'away_win') & (previous_matches['away_team'] == home_team))
    ])
    away_wins = len(previous_matches[
        ((previous_matches['results'] == 'home_win') & (previous_matches['home_team'] == away_team)) |
        ((previous_matches['results'] == 'away_win') & (previous_matches['away_team'] == away_team))
    ])
    draws = len(previous_matches[previous_matches['results'] == 'draw'])
    h2h_dict[(home_team, away_team)] = (home_wins, away_wins, draws)

In [46]:
# predict match
def predict_match(home_team, away_team):
    home_rank = rankings[home_team]
    away_rank = rankings[away_team]
    rank_difference = home_rank - away_rank
    neutral = 1
    tournament_encode = 9 #filtered_df[filtered_df['tournament'] == 'FIFA World Cup']['tournament_encode'].iloc[0]

    home_form = team_form[home_team]
    away_form = team_form[away_team]

    if (home_team, away_team) in h2h_dict:
        h2h_home_win, h2h_away_win, h2h_draw = h2h_dict[(home_team, away_team)]
    elif (away_team, home_team) in h2h_dict:
        h2h_away_win, h2h_home_win, h2h_draw = h2h_dict[(away_team, home_team)]
    else:
        h2h_home_win, h2h_away_win, h2h_draw = 0, 0, 0

    x = [[
        neutral, tournament_encode, h2h_home_win, 
        h2h_away_win, h2h_draw, home_form, away_form,
        home_rank, away_rank, rank_difference
    ]]

    probability = model_xgb.predict_proba(x)[0]
    results = {
        'home_win': probability[2],
        'draw': probability[1],
        'away_win': probability[0]
    }

    return results


In [47]:
# simulate group
def simulate_group(teams):
    team_pairs = list(combinations(teams, 2))

    standings = {team: {'match_played': 0, 'win': 0, 'draw': 0, 'lose': 0, 'points': 0} for team in teams}
    
    for home_team, away_team in team_pairs:
        result = predict_match(home_team, away_team)
        probs = np.array(list(result.values()))
        probs = probs / probs.sum()
        outcome = np.random.choice(list(result.keys()), p=probs)

        standings[home_team]['match_played'] += 1
        standings[away_team]['match_played'] += 1

        if outcome == 'home_win':
            standings[home_team]['points'] += 3
            standings[home_team]['win'] += 1
            standings[away_team]['lose'] += 1

        elif outcome == 'draw':
            standings[home_team]['points'] += 1
            standings[home_team]['draw'] += 1
            standings[away_team]['points'] += 1
            standings[away_team]['draw'] += 1

        else:
            standings[away_team]['points'] += 3
            standings[away_team]['win'] += 1
            standings[home_team]['lose'] += 1

    sorted_standings = sorted(standings, key=lambda team: standings[team]['points'], reverse=True)
    qualifiers = sorted_standings[:2]

    return sorted_standings, qualifiers, standings

In [48]:
# group stage
def group_stage():
    winners = []
    runners_up = []
    third_place = []

    for group_name, teams in groups.items():
        sorted_standings, qualifiers, standings = simulate_group(teams)

        winners.append(qualifiers[0])
        runners_up.append(qualifiers[1])

        third_place_team = sorted_standings[2]
        third_place.append({
            'team': third_place_team,
            'points': standings[third_place_team]['points'],
            'group': group_name
        })

    sorted_third_place = sorted(third_place, key=lambda x: x['points'], reverse=True)
    third_place = sorted_third_place[:8]

    return winners, runners_up, third_place

In [49]:
# retrieve winners, runners_up, and third place
def round32(winners, runners_up, third_place):
    group_names = list(groups.keys())
    winners_dict = dict(zip(group_names, winners))
    runnersup_dict = dict(zip(group_names, runners_up))
    third_place_dict = {tp['group']: tp['team'] for tp in third_place}

    used_third_place = set()

    def get_third_place(possible_groups, third_place_dict):
        for group in possible_groups:
            if group in third_place_dict and third_place_dict[group] not in used_third_place:
                used_third_place.add(third_place_dict[group])
                return third_place_dict[group]
            
        for group, team in third_place_dict.items():
            if team not in used_third_place:
                used_third_place.add(team)
                return team
            
    matches = [
        (winners_dict['Group E'], get_third_place(['Group A', 'Group B', 'Group C', 'Group D', 'Group F'], third_place_dict)),
        (winners_dict['Group I'], get_third_place(['Group C', 'Group D', 'Group F', 'Group G', 'Group H'], third_place_dict)),
        (runnersup_dict['Group A'], runnersup_dict['Group B']),
        (winners_dict['Group F'], runnersup_dict['Group C']),
        (runnersup_dict['Group K'], runnersup_dict['Group L']),
        (winners_dict['Group H'], runnersup_dict['Group J']),
        (winners_dict['Group D'], get_third_place(['Group B', 'Group E', 'Group F', 'Group I', 'Group J'], third_place_dict)),
        (winners_dict['Group G'], get_third_place(['Group A', 'Group E', 'Group H', 'Group I', 'Group J'], third_place_dict)),
        (winners_dict['Group C'], runnersup_dict['Group F']),
        (runnersup_dict['Group E'], runnersup_dict['Group I']),
        (winners_dict['Group A'], get_third_place(['Group C', 'Group E', 'Group F', 'Group H', 'Group I'], third_place_dict)),
        (winners_dict['Group L'], get_third_place(['Group E', 'Group H', 'Group I', 'Group J', 'Group K'], third_place_dict)),
        (winners_dict['Group J'], runnersup_dict['Group H']),
        (runnersup_dict['Group D'], runnersup_dict['Group G']),
        (winners_dict['Group B'], get_third_place(['Group E', 'Group F', 'Group G', 'Group I', 'Group J'], third_place_dict)),
        (winners_dict['Group K'], get_third_place(['Group D', 'Group E', 'Group I', 'Group J', 'Group L'], third_place_dict))
    ]

    return matches
    

In [50]:
# knockout stage
def simulate_knockout(matches):
    winners = []

    for home_team, away_team in matches:
        result = predict_match(home_team, away_team)
        probs = np.array(list(result.values()))
        probs = probs / probs.sum()
        outcome = np.random.choice(list(result.keys()), p=probs)

        if outcome == 'draw':
            home_prob = float(result['home_win']) + float(result['draw']) / 2
            away_prob = float(result['away_win']) + float(result['draw']) / 2
            total = home_prob + away_prob
            outcome = np.random.choice(['home_win', 'away_win'], p=[home_prob/total, away_prob/total])
        
        if outcome == 'home_win':
            winners.append(home_team)
        else:
            winners.append(away_team)

    return winners

In [51]:
# simulate overall tournament
def simulate_tournament():
    winners, runners_up, third_place = group_stage()

    matches = round32(winners, runners_up, third_place)    
    r32_qualifiers = [team for match in matches for team in match] # teams that reaches r32
    r16_qualifiers = simulate_knockout(matches) # teams that reaches r16

    matches = [(r16_qualifiers[i], r16_qualifiers[i+1]) for i in range(0, len(r16_qualifiers), 2)]
    qf_qualifiers = simulate_knockout(matches) # teams that reaches qf

    matches = [(qf_qualifiers[i], qf_qualifiers[i+1]) for i in range(0, len(qf_qualifiers), 2)]
    sf_qualifiers = simulate_knockout(matches) # teams that reaches sf

    matches = [(sf_qualifiers[i], sf_qualifiers[i+1]) for i in range(0, len(sf_qualifiers), 2)]
    finalists = simulate_knockout(matches) # teams that reaches the final

    matches = [(finalists[0], finalists[1])]
    champion = simulate_knockout(matches)[0] # winner
    
    return r32_qualifiers, r16_qualifiers, qf_qualifiers, sf_qualifiers, finalists, champion

In [52]:
advancement_counts = {
    team: {"R32": 0, "R16": 0, "QF": 0, "SF": 0, "Final": 0, "Champion": 0} 
    for team in all_teams
}

for i in range(10000):
    r32_qualifiers, r16_qualifiers, qf_qualifiers, sf_qualifiers, finalists, champion = simulate_tournament()
    
    for team in r32_qualifiers:
        advancement_counts[team]["R32"] += 1

    for team in r16_qualifiers:
        advancement_counts[team]["R16"] += 1

    for team in qf_qualifiers:
        advancement_counts[team]["QF"] += 1

    for team in sf_qualifiers:
        advancement_counts[team]["SF"] += 1

    for team in finalists:
        advancement_counts[team]["Final"] += 1

    advancement_counts[champion]["Champion"] += 1

advancement_probabilities = {
    team: {round: count/10000*100 for round, count in rounds.items()}
    for team, rounds in advancement_counts.items()
}

print(advancement_probabilities)


{'Mexico': {'R32': 91.79, 'R16': 60.089999999999996, 'QF': 27.689999999999998, 'SF': 12.72, 'Final': 5.72, 'Champion': 2.29}, 'South Africa': {'R32': 50.01, 'R16': 15.09, 'QF': 3.95, 'SF': 0.79, 'Final': 0.13999999999999999, 'Champion': 0.05}, 'South Korea': {'R32': 79.14, 'R16': 41.03, 'QF': 15.97, 'SF': 5.65, 'Final': 2.2399999999999998, 'Champion': 0.75}, 'Czechia': {'R32': 61.019999999999996, 'R16': 23.66, 'QF': 7.430000000000001, 'SF': 2.17, 'Final': 0.61, 'Champion': 0.18}, 'Canada': {'R32': 85.79, 'R16': 41.24, 'QF': 16.86, 'SF': 5.57, 'Final': 1.69, 'Champion': 0.48}, 'Bosnia and Herzegovina': {'R32': 48.79, 'R16': 11.68, 'QF': 2.82, 'SF': 0.48, 'Final': 0.1, 'Champion': 0.01}, 'Qatar': {'R32': 58.14, 'R16': 17.73, 'QF': 4.84, 'SF': 0.86, 'Final': 0.13999999999999999, 'Champion': 0.03}, 'Switzerland': {'R32': 87.83999999999999, 'R16': 49.5, 'QF': 22.66, 'SF': 8.93, 'Final': 3.71, 'Champion': 1.44}, 'Brazil': {'R32': 94.73, 'R16': 56.07, 'QF': 35.55, 'SF': 20.16, 'Final': 11.76,

In [53]:
import json

with open("frontend/src/data/advancement_probabilities.json", "w", encoding="utf-8") as json_file:
    json.dump(advancement_probabilities, json_file, indent=4)